# Day 2 - Data Engineering
This notebook covers loading raw Kaggle datasets, preprocessing, combining, feature engineering, class imbalance mapping, and exporting the final structured dataframe for predictive ML stages.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql.window import Window

try:
    spark
    print("Spark session already active (Databricks).")
except NameError:
    print("Initializing local Spark session.")
    spark = SparkSession.builder \
        .appName("DataEngineeringDay2") \
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
        .getOrCreate()

data_path = "../data/" 

inpatient_df = spark.read.csv(data_path + "Inpatient_data.csv", header=True, inferSchema=True)
outpatient_df = spark.read.csv(data_path + "Outpatient_data.csv", header=True, inferSchema=True)
beneficiary_df = spark.read.csv(data_path + "Beneficiary_data.csv", header=True, inferSchema=True)
train_df = spark.read.csv(data_path + "Train-1542865627584.csv", header=True, inferSchema=True)

print("Datasets successfully loaded.")

In [ ]:
# Recode PotentialFraud from Yes/No strings to 1/0 integers
train_df = train_df.withColumn(
    "PotentialFraud", 
    F.when(F.col("PotentialFraud") == "Yes", 1).otherwise(0)
)

# Recode all 11 ChronicCond_* columns from 1/2 to std 1/0 binary flags
chronic_cols = [c for c in beneficiary_df.columns if c.startswith("ChronicCond_")]
for c in chronic_cols:
    beneficiary_df = beneficiary_df.withColumn(
        c,
        F.when(F.col(c) == 1, 1).when(F.col(c) == 2, 0).otherwise(None)
    )

# Recode RenalDiseaseIndicator from "Y"/"0" to 1/0
beneficiary_df = beneficiary_df.withColumn(
    "RenalDiseaseIndicator",
    F.when(F.col("RenalDiseaseIndicator") == "Y", 1).otherwise(0)
).withColumnRenamed("RenalDiseaseIndicator", "HasRenalDisease")

In [ ]:
# Append a claim_type flag
inpatient_df = inpatient_df.withColumn("claim_type", F.lit(1))
outpatient_df = outpatient_df.withColumn("claim_type", F.lit(0))

# UNION the inpatient and outpatient DataFrames
# allowMissingColumns is crucial since IP has procedures and OP does not
claims_df = inpatient_df.unionByName(outpatient_df, allowMissingColumns=True)

# LEFT JOIN the unified claims DataFrame to Beneficiary on BeneID
claims_df = claims_df.join(beneficiary_df, on="BeneID", how="left")

# LEFT JOIN the resulting DataFrame to Train on Provider
claims_df = claims_df.join(train_df, on="Provider", how="left")

print("Master claim DataFrame shape:", (claims_df.count(), len(claims_df.columns)))

In [ ]:
# Impute DeductibleAmtPaid using median imputation for inpatient claims
median_deductible = claims_df.filter(F.col("claim_type") == 1) \
    .approxQuantile("DeductibleAmtPaid", [0.5], 0.01)[0]

claims_df = claims_df.withColumn(
    "DeductibleAmtPaid",
    F.when((F.col("claim_type") == 1) & (F.col("DeductibleAmtPaid").isNull()), median_deductible)
     .otherwise(F.col("DeductibleAmtPaid"))
)

# Fill missing ClmDiagnosisCode_1 with UNKNOWN
claims_df = claims_df.withColumn(
    "ClmDiagnosisCode_1",
    F.when(F.col("ClmDiagnosisCode_1").isNull(), "UNKNOWN").otherwise(F.col("ClmDiagnosisCode_1"))
)

# Derive DiagnosisCodeCount by counting non-nulls
diag_cols = [f"ClmDiagnosisCode_{i}" for i in range(1, 10)]
claims_df = claims_df.withColumn(
    "DiagnosisCodeCount",
    sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in diag_cols if c in claims_df.columns)
)

# Derive ProcedureCodeCount by counting non-nulls
proc_cols = [f"ClmProcedureCode_{i}" for i in range(1, 7)]
claims_df = claims_df.withColumn(
    "ProcedureCodeCount",
    sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in proc_cols if c in claims_df.columns)
)

# Derive PhysicianCount by evaluating phys columns
phys_cols = ["AttendingPhysician", "OperatingPhysician", "OtherPhysician"]
claims_df = claims_df.withColumn(
    "PhysicianCount",
    sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in phys_cols)
)
claims_df = claims_df.withColumn("HasOperatingPhysician", F.when(F.col("OperatingPhysician").isNotNull(), 1).otherwise(0))
claims_df = claims_df.withColumn("HasOtherPhysician", F.when(F.col("OtherPhysician").isNotNull(), 1).otherwise(0))

# Derive LengthOfStay and ClaimDuration
claims_df = claims_df.withColumn("ClaimStartDt", F.to_date(F.col("ClaimStartDt"), "yyyy-MM-dd"))
claims_df = claims_df.withColumn("ClaimEndDt", F.to_date(F.col("ClaimEndDt"), "yyyy-MM-dd"))
claims_df = claims_df.withColumn("AdmissionDt", F.to_date(F.col("AdmissionDt"), "yyyy-MM-dd"))
claims_df = claims_df.withColumn("DischargeDt", F.to_date(F.col("DischargeDt"), "yyyy-MM-dd"))
claims_df = claims_df.withColumn("DOB", F.to_date(F.col("DOB"), "yyyy-MM-dd"))
claims_df = claims_df.withColumn("DOD", F.to_date(F.col("DOD"), "yyyy-MM-dd"))

claims_df = claims_df.withColumn("ClaimDuration", F.datediff(F.col("ClaimEndDt"), F.col("ClaimStartDt")))
claims_df = claims_df.withColumn("LengthOfStay", F.datediff(F.col("DischargeDt"), F.col("AdmissionDt")))

# Compute Age, IsDeceased, HasPrimaryDiagnosis, ICD_Chapter
claims_df = claims_df.withColumn("Age", F.datediff(F.col("ClaimStartDt"), F.col("DOB")) / 365.25)
claims_df = claims_df.withColumn("IsDeceased", F.when(F.col("DOD").isNotNull(), 1).otherwise(0))
claims_df = claims_df.withColumn("ChronicConditionCount", sum(F.col(c) for c in chronic_cols))
claims_df = claims_df.withColumn("HasPrimaryDiagnosis", F.when(F.col("ClmDiagnosisCode_1") != "UNKNOWN", 1).otherwise(0))
claims_df = claims_df.withColumn("ICD_Chapter", F.substring("ClmDiagnosisCode_1", 1, 3))
claims_df = claims_df.withColumn("MedicareCoverageMonths", F.col("NoOfMonths_PartACov") + F.col("NoOfMonths_PartBCov"))

claims_df = claims_df.withColumn("IPtoOPReimbursementRatio", F.col("IPAnnualReimbursementAmt") / (F.col("OPAnnualReimbursementAmt") + 1))


In [ ]:
# Compute class_weight natively based on inverse frequencies
total_count = claims_df.count()
fraud_count = claims_df.filter(F.col("PotentialFraud") == 1).count()
non_fraud_count = total_count - fraud_count

weight_fraud = total_count / (2.0 * fraud_count) if fraud_count > 0 else 1.0
weight_non_fraud = total_count / (2.0 * non_fraud_count) if non_fraud_count > 0 else 1.0

claims_df = claims_df.withColumn(
    "class_weight",
    F.when(F.col("PotentialFraud") == 1, weight_fraud).otherwise(weight_non_fraud)
)

print(f"Weights generated. Fraud weight: {weight_fraud:.2f}, Non-Fraud weight: {weight_non_fraud:.2f}")

In [ ]:
# Identify and drop explicitly non-predictive or highly-null categorical identifiers
cols_to_drop = [
    "AttendingPhysician", "OperatingPhysician", "OtherPhysician", 
    "DOB", "DOD", "DiagnosisGroupCode", "ClmAdmitDiagnosisCode",
    "ClmDiagnosisCode_10", "County", "IPAnnualDeductibleAmt", 
    "OPAnnualDeductibleAmt", "NoOfMonths_PartACov", "NoOfMonths_PartBCov",
    "IPAnnualReimbursementAmt", "OPAnnualReimbursementAmt"
]

# Extending drop list with procedure code arrays
cols_to_drop.extend(proc_cols)
cols_to_drop.extend(diag_cols)

cols_missing_handled = []
for c in claims_df.columns:
    if c in cols_to_drop:
        cols_missing_handled.append(c)

claims_df = claims_df.drop(*cols_missing_handled)

# Check summary statistics
claims_df.printSchema()

# Save the engineered DataFrame as a Parquet or Delta table
output_path = "../data/day2_engineered_features"
print(f"Writing output data to: {output_path}...")
claims_df.write.mode("overwrite").parquet(output_path)
print("Feature Engineering notebook completed successfully. Exported for Day 3 Mode Training.")